In [2]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Model selection
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Load dataset
data = pd.read_csv("abc.csv")

# Display first 5 rows
data.head()

,City,Household_Size,Seasonal_Index,Garden_Area,Avg_Rainfall,Household_Income,Previous_Day_Usage,City_Population,Daily_Water_Used,Avg_Water_Table,Suitability
0,Pune,3.0,1.105,179.345,650.0,66700.0,271.400,7300000.0,266.357,520.0,Suitable
1,Chennai,4.0,0.500,93.100,1400.0,69900.0,125.583,11000000.0,122.500,1120.0,Suitable
2,Jaipur,2.0,NaN,19.800,550.0,97900.0,124.016,3500000.0,NaN,440.0,Suitable
3,Kolkata,2.0,1.050,44.330,1600.0,40000.0,219.800,14800000.0,219.946,1280.0,Suitable
4,Pune,1.0,1.140,136.012,650.0,82300.0,256.036,7300000.0,257.690,520.0,Suitable


In [4]:
# Check shape (rows, columns)
print("Shape:", data.shape)
data.info()
data.describe()
data.isnull().sum()

Shape: (200000, 11)
<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   City                200000 non-null  str    
 1   Household_Size      189851 non-null  float64
 2   Seasonal_Index      190039 non-null  float64
 3   Garden_Area         189882 non-null  float64
 4   Avg_Rainfall        190025 non-null  float64
 5   Household_Income    189881 non-null  float64
 6   Previous_Day_Usage  190170 non-null  float64
 7   City_Population     190088 non-null  float64
 8   Daily_Water_Used    189915 non-null  float64
 9   Avg_Water_Table     189987 non-null  float64
 10  Suitability         190088 non-null  str    
dtypes: float64(9), str(2)
memory usage: 16.8 MB


City                      0
Household_Size        10149
Seasonal_Index         9961
Garden_Area           10118
Avg_Rainfall           9975
Household_Income      10119
Previous_Day_Usage     9830
City_Population        9912
Daily_Water_Used      10085
Avg_Water_Table       10013
Suitability            9912
dtype: int64

In [5]:
# Fill numerical columns with median
data.fillna(data.median(numeric_only=True), inplace=True)
# Fill categorical column with mode
data["Suitability"] = data["Suitability"].fillna(data["Suitability"].mode()[0])
data.isnull().sum()

City                  0
Household_Size        0
Seasonal_Index        0
Garden_Area           0
Avg_Rainfall          0
Household_Income      0
Previous_Day_Usage    0
City_Population       0
Daily_Water_Used      0
Avg_Water_Table       0
Suitability           0
dtype: int64

In [6]:
print(data.columns)

Index(['City', 'Household_Size', 'Seasonal_Index', 'Garden_Area',
       'Avg_Rainfall', 'Household_Income', 'Previous_Day_Usage',
       'City_Population', 'Daily_Water_Used', 'Avg_Water_Table',
       'Suitability'],
      dtype='str')


In [7]:
data = pd.get_dummies(data, columns=["City"], drop_first=True)
print(data.columns)

Index(['Household_Size', 'Seasonal_Index', 'Garden_Area', 'Avg_Rainfall',
       'Household_Income', 'Previous_Day_Usage', 'City_Population',
       'Daily_Water_Used', 'Avg_Water_Table', 'Suitability', 'City_Bangalore',
       'City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Jaipur',
       'City_Kolkata', 'City_Lucknow', 'City_Mumbai', 'City_Pune'],
      dtype='str')


In [8]:
# Format numerical columns

data["Household_Size"] = data["Household_Size"].round().astype(int)

data["Seasonal_Index"] = data["Seasonal_Index"].round(3)
data["Garden_Area"] = data["Garden_Area"].round(3)
data["Previous_Day_Usage"] = data["Previous_Day_Usage"].round(3)
data["Daily_Water_Used"] = data["Daily_Water_Used"].round(3)

# Round income to nearest 100
data["Household_Income"] = (data["Household_Income"] / 100).round() * 100

In [9]:
import numpy as np

# Select numerical columns
num_cols = data.select_dtypes(include=np.number).columns

# Calculate Q1, Q3, IQR
Q1 = data[num_cols].quantile(0.25)
Q3 = data[num_cols].quantile(0.75)
IQR = Q3 - Q1

# Remove outliers
data = data[~((data[num_cols] < (Q1 - 1.5 * IQR)) | 
              (data[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

# Check shape
print("Shape after outlier removal:", data.shape)

Shape after outlier removal: (196002, 19)


In [10]:


# 6. Feature and Target Split

X = data.drop(["Daily_Water_Used", "Suitability", "Avg_Water_Table"], axis=1)
y = data["Daily_Water_Used"]


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(156801, 16)
(39201, 16)


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data
X_train = scaler.fit_transform(X_train)

# Transform test data
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)


(156801, 16)
(39201, 16)


In [13]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [14]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(y_pred_rf)

[156.43603 241.92143 110.23108 ... 135.71345 207.8886  143.81503]
